# Velocity-space separation: NEO vs MBA by antisun longitude bin

(v_λ, v_β) contours for NEOs (blue) and MBAs (orange) across all 29 Δλ_⊙ bins. Data: `outputs/phase2/sorcha_comparison_v5_masked.parquet`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import gaussian_filter
%matplotlib inline

df = pd.read_parquet("outputs/phase2/sorcha_comparison_v5_masked.parquet",
                     columns=["ecl_lon", "mjd0_tai", "population", "vlam", "vbeta"])

# Antisun-relative longitude for each tracklet
T        = (df.mjd0_tai.values + 2400000.5) - 2451545.0
lam_sun  = (280.46 + 0.9856474 * T) % 360.0
anti     = (lam_sun + 180.0) % 360.0
df["dlon"] = ((df.ecl_lon.values - anti + 180) % 360) - 180

# Grid definition (same as the VDP map grid)
GLON = np.arange(-140, 141, 10)   # 29 bin centres
HALF = 5.0                         # half-width of each bin

print(f"{len(df):,} tracklets | {len(GLON)} longitude bins")
print("populations:", df.population.value_counts().to_dict())

In [ ]:

#  velocity axis range 
VLIM = 2.2          # deg/day — crop to +-VLIM for both axes
BINS = 80           # resolution of the 2D histogram
SIGMA = 1.5         # Gaussian smoothing (bins) for contours
LEVELS = 6          # number of contour levels per population

edges = np.linspace(-VLIM, VLIM, BINS + 1)
cx = (edges[:-1] + edges[1:]) / 2

pop_neo  = df.population.values == "NEO"
pop_mba  = df.population.values == "MBA"

#  figure layout: 5 rows × 6 cols (last panel empty) 
NCOLS, NROWS = 6, 5
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(18, 15),
                          sharex=True, sharey=True)
axes_flat = axes.flatten()

for idx, dlon_cen in enumerate(GLON):
    ax = axes_flat[idx]
    sel = np.abs(df["dlon"].values - dlon_cen) < HALF

    for mask, color, label in [
        (sel & pop_mba, "tab:orange", "MBA"),
        (sel & pop_neo, "tab:blue",   "NEO"),
    ]:
        vl = df.vlam.values[mask]
        vb = df.vbeta.values[mask]
        if mask.sum() < 20:
            continue
        H, _, _ = np.histogram2d(vl, vb, bins=edges)
        H = gaussian_filter(H / (H.sum() + 1e-30), sigma=SIGMA)
        vmax = H.max()
        lvls = np.logspace(np.log10(vmax * 0.02), np.log10(vmax * 0.90), LEVELS)
        ax.contourf(cx, cx, H.T, levels=lvls, colors=[color], alpha=0.35)
        ax.contour( cx, cx, H.T, levels=[lvls[0], lvls[-1]],
                    colors=[color], linewidths=[0.5, 1.2])

    n_neo = (sel & pop_neo).sum()
    n_mba = (sel & pop_mba).sum()
    ax.set_title(f"Δλ={dlon_cen:+.0f}°\nN_NEO={n_neo:,} N_MBA={n_mba:,}",
                 fontsize=7.5)
    ax.axhline(0, color="0.6", lw=0.4, ls=":")
    ax.axvline(0, color="0.6", lw=0.4, ls=":")
    ax.set_xlim(-VLIM, VLIM)
    ax.set_ylim(-VLIM, VLIM)

# hide the unused 30th panel
axes_flat[len(GLON)].set_visible(False)

# shared axis labels
for ax in axes[-1]:
    ax.set_xlabel(r"$v_\lambda$ (deg/day)", fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel(r"$v_\beta$ (deg/day)", fontsize=8)

# legend
from matplotlib.patches import Patch
axes_flat[0].legend(handles=[Patch(color="tab:blue",   label="NEO"),
                              Patch(color="tab:orange", label="MBA")],
                    fontsize=7, loc="upper right")

fig.suptitle(r"Velocity-space separation: NEO (blue) vs MBA (orange) per antisun-$\Delta\lambda_\odot$ bin",
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig("Figures/velocity_separation_by_lon.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved Figures/velocity_separation_by_lon.png")
